In [ ]:
# ¿Cómo podemos usar probabilidades de incumplimiento para decidir qué solicitudes de crédito revisar primero?

# El archivo contiene, para cada solicitud, la probabilidad de incumplimiento estimada por un modelo
# y si el cliente realmente incumplió.

import pandas as pd

dataframe = pd.read_csv("../data/risk_predictions.csv")

In [ ]:
# ¿Cuántas solicitudes están disponibles para evaluar las probabilidades?

dataframe.shape

In [ ]:
# Se inspeccionan las variables de cada solicitud.

dataframe.head()

In [ ]:
# ¿Qué proporción de solicitudes incumplió realmente?

dataframe.observed_default.value_counts(normalize=True)

In [ ]:
# Si las probabilidades fueran correctas en promedio, su media se parecería a la tasa observada.

dataframe.observed_default.mean(), dataframe.default_probability.mean()

In [ ]:
# ¿Las solicitudes que incumplieron recibieron probabilidades más altas?

import matplotlib.pyplot as plt

plt.figure()
plt.hist(
    dataframe[dataframe.observed_default == 0]["default_probability"],
    bins=30,
    alpha=0.6,
    label="No incumplió",
)
plt.hist(
    dataframe[dataframe.observed_default == 1]["default_probability"],
    bins=30,
    alpha=0.6,
    label="Incumplió",
)
plt.xlabel("Probabilidad de incumplimiento")
plt.ylabel("Número de solicitudes")
plt.legend()
plt.show()

In [ ]:
# El AUC mide qué tan bien ordenan las probabilidades: es la probabilidad de que una solicitud
# que incumplió reciba una probabilidad mayor que una que no incumplió (0.5 = azar, 1 = perfecto).

from sklearn.metrics import roc_auc_score

roc_auc_score(
    y_true=dataframe.observed_default,
    y_score=dataframe.default_probability,
)

In [ ]:
# Calibración: se agrupan las solicitudes por rangos de probabilidad y, en cada rango,
# se compara la probabilidad promedio estimada con la tasa de incumplimiento observada.

import numpy as np

dataframe["probability_bin"] = pd.cut(
    dataframe.default_probability,
    bins=np.linspace(0, 1, 11),
    include_lowest=True,
)

calibration_summary = (
    dataframe.groupby("probability_bin", observed=True)
    .agg(
        applications=("observed_default", "size"),
        mean_predicted_probability=("default_probability", "mean"),
        observed_default_rate=("observed_default", "mean"),
    )
    .reset_index()
)

calibration_summary

In [ ]:
# Si las probabilidades estuvieran bien calibradas, los puntos quedarían sobre la diagonal.

plt.figure()
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Calibración perfecta")
plt.plot(
    calibration_summary.mean_predicted_probability,
    calibration_summary.observed_default_rate,
    marker="o",
    color="tab:green",
    label="Modelo",
)
plt.xlabel("Probabilidad promedio estimada")
plt.ylabel("Tasa de incumplimiento observada")
plt.legend()
plt.show()

In [ ]:
# Priorización: las solicitudes se ordenan de mayor a menor probabilidad y se revisan primero
# las de mayor riesgo.

dataframe.sort_values("default_probability", ascending=False).head(10)

In [ ]:
# Cada umbral define qué solicitudes se revisan (probabilidad >= umbral).
#
# Casos tipicos:
#   umbral bajo: se revisan casi todas y se detectan casi todos los incumplimientos.
#   umbral alto: se revisan pocas, con más aciertos, pero se escapan muchos incumplimientos.
#
# precision: de las solicitudes revisadas, qué proporción incumplió.
# recall: de todos los incumplimientos, qué proporción quedó dentro de las revisadas.

from sklearn.metrics import precision_score, recall_score

thresholds = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

rows = []
for threshold in thresholds:
    reviewed = (dataframe.default_probability >= threshold).astype(int)

    rows.append(
        {
            "threshold": threshold,
            "reviewed_applications": reviewed.sum(),
            "reviewed_share": reviewed.mean(),
            "precision": precision_score(
                y_true=dataframe.observed_default,
                y_pred=reviewed,
                zero_division=0,
            ),
            "recall": recall_score(
                y_true=dataframe.observed_default,
                y_pred=reviewed,
            ),
        }
    )

threshold_tradeoff = pd.DataFrame(rows)
threshold_tradeoff

In [ ]:
# Al subir el umbral se gana precisión pero se pierde recall.

plt.figure()
plt.plot(threshold_tradeoff.threshold, threshold_tradeoff.precision, marker="o", label="Precision")
plt.plot(threshold_tradeoff.threshold, threshold_tradeoff.recall, marker="o", label="Recall")
plt.plot(
    threshold_tradeoff.threshold,
    threshold_tradeoff.reviewed_share,
    marker="o",
    label="Proporción revisada",
)
plt.xlabel("Umbral")
plt.legend()
plt.show()

In [ ]:
# Se supone que el equipo solo puede revisar el 20 % de las solicitudes:
# el umbral es la probabilidad que deja por encima al 20 % más riesgoso.

review_capacity = 0.2

selected_threshold = dataframe.default_probability.quantile(1 - review_capacity)
dataframe["reviewed"] = (dataframe.default_probability >= selected_threshold).astype(int)

selected_threshold, dataframe.reviewed.mean()

In [ ]:
# Revisión por grupo: ¿la priorización trata de forma parecida a los dos valores de sex?
#
# Se comparan, para cada grupo, la tasa observada, la probabilidad promedio,
# la proporción revisada y qué tanto de sus incumplimientos se detecta.

rows = []
for group in sorted(dataframe.sex.unique()):
    group_data = dataframe[dataframe.sex == group]

    rows.append(
        {
            "sex": group,
            "applications": len(group_data),
            "observed_default_rate": group_data.observed_default.mean(),
            "mean_predicted_probability": group_data.default_probability.mean(),
            "reviewed_share": group_data.reviewed.mean(),
            "precision": precision_score(
                y_true=group_data.observed_default,
                y_pred=group_data.reviewed,
                zero_division=0,
            ),
            "recall": recall_score(
                y_true=group_data.observed_default,
                y_pred=group_data.reviewed,
            ),
            "auc": roc_auc_score(
                y_true=group_data.observed_default,
                y_score=group_data.default_probability,
            ),
        }
    )

group_review = pd.DataFrame(rows)
group_review

In [ ]:
# Se guardan las tablas de calibración, umbrales y revisión por grupo como evidencia verificable.

calibration_summary.to_csv("../submission/calibration_summary.csv", index=False)
threshold_tradeoff.to_csv("../submission/threshold_tradeoff.csv", index=False)
group_review.to_csv("../submission/group_review.csv", index=False)

In [ ]:
# Se comprueba que las tablas almacenadas se pueden leer de nuevo.

pd.read_csv("../submission/group_review.csv")